# 05 · Mini agent-shaped workflow

**Estimated study time: 12 minutes.**

You already know programming from Java. Here we focus on what feels different in Python. Run the cells in order, predict the results, and change one value before moving on.

## Scenario: a helpdesk assistant that proposes a reply

We will run an **agent-shaped deterministic workflow**, not an autonomous LLM agent. A fake knowledge-base lookup simulates I/O. We validate input, categorize a ticket, retrieve suggestions concurrently, prepare a response, and require human approval before simulated sending. No API keys.

In [ ]:
from mini_agent import Ticket, prepare_reply, simulate_send

ticket = Ticket(id=101, subject="Cannot sign in", body="I can't log in to my account")
proposal = await prepare_reply(ticket)
print(proposal.model_dump())
assert proposal.approved is False

## Human-in-the-loop

The proposal is intentionally never sent just because an assistant generated it. Approval is a separate step. This design is useful whenever tools could change external state.

In [ ]:
rejected = simulate_send(proposal, approved=False)
print(rejected)
assert rejected["would_send"] is False
approved = simulate_send(proposal, approved=True)
print(approved)
assert approved["would_send"] is True

## Inspect the implementation

Open `src/mini_agent/models.py`, `tools.py` and `workflow.py`. Observe the separation between data contracts, I/O simulation and orchestration.

In [ ]:
import inspect
from mini_agent import prepare_reply
print(inspect.signature(prepare_reply))

### Challenge

Create a ticket about a missing invoice. Predict which category is assigned and why. Try an unrelated ticket too. Explain where you would later insert an LLM and how you would test its outputs.

In [ ]:
invoice = Ticket(id=102, subject="Invoice issue", body="Cannot download my invoice")
other = Ticket(id=103, subject="Suggestion", body="Add dark mode")
invoice_reply = await prepare_reply(invoice)
other_reply = await prepare_reply(other)
print(invoice_reply.category, other_reply.category)
assert invoice_reply.category == "billing"